# 02a — Build & Validate the Agent

Imports the shared `agent_lib.py` module, unit-tests each of the three tools directly (without an LLM), then assembles the ReAct agent and executes a smoke-test invocation.

### Validation Steps

- Import the shared `agent_lib.py` library
- Unit-test semantic retrieval against the Vector Search index
- Unit-test conflict checking against `lexpath_conflicts`
- Unit-test case routing through `lexpath_routing_schema`
- Assemble the full ReAct agent
- Execute a single end-to-end smoke test to verify tool orchestration

### Notes

Databricks notebooks do not share Python runtime state. As a result, this notebook focuses solely on build validation. The downstream notebooks (`02b_run_agent` and `03_evaluation`) independently reconstruct the agent from the same shared library to ensure reproducibility and avoid cross-notebook dependency issues.

In [0]:
# Configure Widgets
dbutils.widgets.text("catalog", "workspace", "Unity Catalog name")
dbutils.widgets.text("schema", "default", "Schema")
dbutils.widgets.text("vs_endpoint", "lexpath_vs_endpoint", "Vector Search Endpoint")
dbutils.widgets.text("llm_endpoint", "anthropic-claude-sonnet-4-6", "LLM Serving Endpoint")

In [0]:
# Install LangChain/Databricks/Vector Search/MLflow Stack — pinned for reproducibility
# Uninstall to ensure clean state
%pip uninstall -y langgraph langgraph-checkpoint langgraph-prebuilt langgraph-sdk
# Install fresh
%pip install --upgrade langgraph databricks-langchain databricks-vectorsearch==0.75

Found existing installation: langgraph 1.2.6
Uninstalling langgraph-1.2.6:
  Successfully uninstalled langgraph-1.2.6
Found existing installation: langgraph-checkpoint 4.1.1
Uninstalling langgraph-checkpoint-4.1.1:
  Successfully uninstalled langgraph-checkpoint-4.1.1
Found existing installation: langgraph-prebuilt 1.1.0
Uninstalling langgraph-prebuilt-1.1.0:
  Successfully uninstalled langgraph-prebuilt-1.1.0
Found existing installation: langgraph-sdk 0.4.2
Uninstalling langgraph-sdk-0.4.2:
  Successfully uninstalled langgraph-sdk-0.4.2
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
  Using cached langgraph-1.2.6-py3-none-any.whl.metadata (4.9 kB)
  Using cached langgraph_checkpoint-4.1.1-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.4.2-py3-none-any.whl.metadata (3.6 kB)
Using cached langgraph-1.2.6-py3-none-any.whl

In [0]:
# Restart Python
dbutils.library.restartPython()

In [0]:
# Import libraries and configure the agent
import sys, os, json
import importlib
sys.path.append(os.getcwd())  # ensure the notebook's workspace folder is importable
 
import mlflow
import agent_lib
importlib.reload(agent_lib)  # Reload to pick up any file changes
 
agent_lib.configure(
    catalog=dbutils.widgets.get("catalog"),
    schema=dbutils.widgets.get("schema"),
    vs_endpoint=dbutils.widgets.get("vs_endpoint"),
)
mlflow.langchain.autolog()

agent_lib configured — index workspace.default.ledgar_provisions_index, 100 routing labels, 10 conflict matters


####Tool 1: Semantic Retrieval

In [0]:
# Unit Test Tool 1: semantic retrieval should return TOP_K grounded provisions
out = json.loads(agent_lib.semantic_retrieval.invoke(
    {"query": "My employer is refusing to pay the severance in my contract"}))
assert len(out) > 0 and {"category", "score", "provision_excerpt"} <= set(out[0]), out
print(f"✅ semantic_retrieval: {len(out)} results, top category = {out[0]['category']}")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
✅ semantic_retrieval: 5 results, top category = Releases


Trace(trace_id=tr-da4f68d775e9add7e580c10d8e056d8f)

####Tool 2: Conflict Check

In [0]:
# Unit Test Tool 2 — conflict check: known client → CONFLICT_FLAG, unknown name → CLEARED
flag = json.loads(agent_lib.conflict_check.invoke({"party_names": "Atlas Manufacturing"}))
clear = json.loads(agent_lib.conflict_check.invoke({"party_names": "Jane Doe"}))
assert flag["result"] == "CONFLICT_FLAG" and clear["result"] == "CLEARED", (flag, clear)
print(f"✅ conflict_check: flagged {len(flag['matches'])} matters; unknown name CLEARED")

✅ conflict_check: flagged 2 matters; unknown name CLEARED


[Trace(trace_id=tr-0f01ffa544942d8eb865395db01a594d), Trace(trace_id=tr-beeefc395e28a09d64acae53da5f3168)]

####Tool 3: Case Routing

In [0]:
# Unit Test Tool 3 — case routing: valid label routes, junk label → Unrouted
routed = json.loads(agent_lib.case_routing.invoke({"category_label": "Governing Laws"}))
junk = json.loads(agent_lib.case_routing.invoke({"category_label": "Not A Real Label"}))
assert routed["recognized"] and not junk["recognized"], (routed, junk)
print(f"✅ case_routing: 'Governing Laws' → {routed['practice_area']}; junk → Unrouted")

✅ case_routing: 'Governing Laws' → Litigation; junk → Unrouted


[Trace(trace_id=tr-8e1aaae9c6dbc061fb8199c787b1a4da), Trace(trace_id=tr-db8bd9620721940705a98e8ff3e31efd)]

In [0]:
# Assemble the Agent
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint")
executor = agent_lib.build_agent(LLM_ENDPOINT, verbose=True)
print(f"Agent assembled on {LLM_ENDPOINT} with {len(agent_lib.tools)} tools")

Agent assembled on anthropic-claude-sonnet-4-6 with 3 tools


In [0]:
# End-to-end test (run the Reason → Act → Observe loop with verbose=True)
# Note: Using the actual endpoint from the widget. Verify the endpoint exists via workspace settings.
try:
    result = executor.invoke({"input":
        "My business partner and I signed an agreement that says disputes go to arbitration, "
        "but now they filed a lawsuit in court instead. I want to enforce the arbitration clause."})
    
    profile = agent_lib.extract_json(result.get("output", ""))
    assert profile.get("status") == "READY_FOR_REVIEW", profile
    assert profile.get("predicted_category"), profile
    print(json.dumps(profile, indent=2))
except Exception as e:
    if "ENDPOINT_NOT_FOUND" in str(e):
        print(f"❌ Endpoint '{LLM_ENDPOINT}' not found.")
        print("\nAvailable endpoints:")
        from databricks.sdk import WorkspaceClient
        w = WorkspaceClient()
        for ep in w.serving_endpoints.list():
            print(f"  - {ep.name}")
        print(f"\nUpdate the 'llm_endpoint' widget to use one of the above endpoints.")
    else:
        raise

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
================================ System Message ================================

You are the client-intake agent for LexPath Legal Group. You process
prospective-client descriptions of legal issues. You are NOT a lawyer and NEVER give
legal advice.
 
Workflow for a legitimate intake:
1. ALWAYS call semantic_retrieval first with the client's description to ground your
   classification in similar legal provisions.
2. Decide the single best LEDGAR category label based on the retrieved provisions.
3. If the intake names specific people or companies, call conflict_check with those
   names. If no parties are named, skip it (conflict_status = "NOT_RUN").
4. Call case_routing with your chosen category label to get the practice area.
5. Respond with ONLY a JSON object (no prose before 

Trace(trace_id=tr-001095cae5d2f06b8901dd807342cea2)

## Build Summary

- All three tools were validated directly against their backing resources:
  - Vector Search index
  - `lexpath_conflicts`
  - `lexpath_routing_schema`
- The agent was assembled via `agent_lib.build_agent()` and successfully smoke-tested end-to-end.
- The resulting trace is available in this notebook's **MLflow Experiment → Traces** tab.
